In [1]:
import os
from pathlib import Path
import warnings
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests

In [2]:
# Suppress FutureWarning from the grangercausalitytests, because it still prints the output when it is not supposed to without using the deprecated paramter verbose=False
warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
parent_dir = Path.cwd().parent.parent

In [4]:
# Import datasets

# Monthly
gig_m_df = pd.read_csv(os.path.join(parent_dir, 'data', 'final', 'gig_monthly_data.csv'))
sp500_m_df = pd.read_csv(os.path.join(parent_dir, 'data', 'final', 'sp500_monthly_data.csv')) 

# Quarterly
gig_q_df = pd.read_csv(os.path.join(parent_dir, 'data', 'final', 'gig_quarterly_data.csv'))
sp500_q_df = pd.read_csv(os.path.join(parent_dir, 'data', 'final', 'sp500_quarterly_data.csv'))

# Annual
gig_a_df = pd.read_csv(os.path.join(parent_dir, 'data', 'final', 'gig_annual_data.csv'))
sp500_a_df = pd.read_csv(os.path.join(parent_dir, 'data', 'final', 'sp500_annual_data.csv'))

In [5]:
columns_to_drop = ['date', 'year', 'month', 'quarter', 'Date', 'Month', 'Quarter', 'Year', 'Company', 'Ticker', 'industryKey', 'sectorKey', 'Close']

In [6]:
# Bonferroni correction for the p-value
monthly_alpha = 0.05 / len([col for col in gig_m_df.columns if col != 'target' and col not in columns_to_drop])
quarterly_alpha = 0.05 / len([col for col in gig_q_df.columns if col != 'target' and col not in columns_to_drop])
annual_alpha = 0.05 / len([col for col in gig_a_df.columns if col != 'target' and col not in columns_to_drop])

In [7]:
# Check granger causality

# Monthly
try:
    for i, col in enumerate(gig_m_df.columns):
        gbl_lst = list(globals().keys())
        if i == 0 and 'gig_m_gr_df' in gbl_lst:
            del gig_m_gr_df
        if col != 'target' and col not in columns_to_drop:
            gr = grangercausalitytests(gig_m_df[[col, 'target']], maxlag=6, verbose=False)
            if 'gig_m_gr_df' not in gbl_lst:
                for k, v in gr.items():
                    if k == 1:
                        gig_m_gr_df = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                    else:
                        df2 = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                        gig_m_gr_df = pd.concat([gig_m_gr_df, df2], ignore_index=True)
            else:
                for k, v in gr.items():
                    df2 = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                    gig_m_gr_df = pd.concat([gig_m_gr_df, df2], ignore_index=True)
    gig_m_gr_df = gig_m_gr_df[gig_m_gr_df['p-value'] < monthly_alpha].reset_index(drop=True)
    gig_m_gr_df = gig_m_gr_df.sort_values(by=['p-value', 'F-statistic'], ascending=[True, False]).reset_index(drop=True)

    iter_gig_m_gr_df = gig_m_gr_df[['column','lag']]
    for i, (col, lag) in enumerate(iter_gig_m_gr_df.itertuples(index=False, name=None)):
        gbl_lst = list(globals().keys())
        gr = grangercausalitytests(gig_m_df[['target', col]], maxlag=6, verbose=False)
        if i == 0:
            if 'gig_m_gr_df_rev' in gbl_lst:
                del gig_m_gr_df_rev
            gig_m_gr_df_rev = pd.DataFrame({'column':col,'lag':lag, 'F-statistic':gr[lag][0]['ssr_ftest'][0], 'p-value':gr[lag][0]['ssr_ftest'][1]}, index=[0])
        else:
            df2 = pd.DataFrame({'column':col,'lag':lag, 'F-statistic':gr[lag][0]['ssr_ftest'][0], 'p-value':gr[lag][0]['ssr_ftest'][1]}, index=[0])
            gig_m_gr_df_rev = pd.concat([gig_m_gr_df_rev, df2], ignore_index=True)
    gig_m_gr_df_rev = gig_m_gr_df_rev[gig_m_gr_df_rev['p-value'] < monthly_alpha].reset_index(drop=True)
    gig_m_gr_df_rev = gig_m_gr_df_rev.sort_values(by=['p-value', 'F-statistic'], ascending=[True, False]).reset_index(drop=True)

    gig_m_gr_df_fin = gig_m_gr_df.merge(gig_m_gr_df_rev, on=['column', 'lag'], how='left', suffixes=('_forward', '_reverse')).reset_index(drop=True)
    gig_m_gr_df_fin = gig_m_gr_df_fin[gig_m_gr_df_fin['p-value_reverse'].isna()].reset_index(drop=True)
    gig_m_gr_df_fin['type'] = 'gig'

    for i, col in enumerate(sp500_m_df.columns):
        gbl_lst = list(globals().keys())
        if i == 0 and 'sp500_m_gr_df' in gbl_lst:
            del sp500_m_gr_df
        if col != 'target' and col not in columns_to_drop:
            gr = grangercausalitytests(sp500_m_df[[col, 'target']], maxlag=6, verbose=False)
            if 'sp500_m_gr_df' not in gbl_lst:
                for k, v in gr.items():
                    if k == 1:
                        sp500_m_gr_df = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                    else:
                        df2 = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                        sp500_m_gr_df = pd.concat([sp500_m_gr_df, df2], ignore_index=True)
            else:
                for k, v in gr.items():
                    df2 = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                    sp500_m_gr_df = pd.concat([sp500_m_gr_df, df2], ignore_index=True)
    sp500_m_gr_df = sp500_m_gr_df[sp500_m_gr_df['p-value'] < monthly_alpha].reset_index(drop=True)
    sp500_m_gr_df = sp500_m_gr_df.sort_values(by=['p-value', 'F-statistic'], ascending=[True, False]).reset_index(drop=True)

    iter_sp500_m_gr_df = sp500_m_gr_df[['column','lag']]
    for i, (col, lag) in enumerate(iter_sp500_m_gr_df.itertuples(index=False, name=None)):
        gbl_lst = list(globals().keys())
        gr = grangercausalitytests(sp500_m_df[['target', col]], maxlag=6, verbose=False)
        if i == 0:
            if 'sp500_m_gr_df_rev' in gbl_lst:
                del sp500_m_gr_df_rev
            sp500_m_gr_df_rev = pd.DataFrame({'column':col,'lag':lag, 'F-statistic':gr[lag][0]['ssr_ftest'][0], 'p-value':gr[lag][0]['ssr_ftest'][1]}, index=[0])
        else:
            df2 = pd.DataFrame({'column':col,'lag':lag, 'F-statistic':gr[lag][0]['ssr_ftest'][0], 'p-value':gr[lag][0]['ssr_ftest'][1]}, index=[0])
            sp500_m_gr_df_rev = pd.concat([sp500_m_gr_df_rev, df2], ignore_index=True)
    sp500_m_gr_df_rev = sp500_m_gr_df_rev[sp500_m_gr_df_rev['p-value'] < monthly_alpha].reset_index(drop=True)
    sp500_m_gr_df_rev = sp500_m_gr_df_rev.sort_values(by=['p-value', 'F-statistic'], ascending=[True, False]).reset_index(drop=True)

    sp500_m_gr_df_fin = sp500_m_gr_df.merge(sp500_m_gr_df_rev, on=['column', 'lag'], how='left', suffixes=('_forward', '_reverse')).reset_index(drop=True)
    sp500_m_gr_df_fin = sp500_m_gr_df_fin[sp500_m_gr_df_fin['p-value_reverse'].isna()].reset_index(drop=True)
    sp500_m_gr_df_fin['type'] = 'sp500'

    monthly_gr_df_fin = pd.concat([gig_m_gr_df_fin, sp500_m_gr_df_fin], ignore_index=True)

except NameError:
    monthly_gr_df_fin = None

monthly_gr_df_fin


,column,lag,F-statistic_forward,p-value_forward,F-statistic_reverse,p-value_reverse,type
0,average_weekly_earnings_data_monthly,5,9.151568,1.724924e-08,NaN,NaN,gig
1,interest_rate_fedfunds_data_monthly,5,7.523936,6.376225e-07,NaN,NaN,gig
2,interest_rate_fedfunds_data_monthly,6,6.453797,1.153554e-06,NaN,NaN,gig
3,interest_rate_fedfunds_data_monthly,4,7.498639,6.167836e-06,NaN,NaN,gig
4,interest_rate_fedfunds_data_monthly,3,8.881894,8.459404e-06,NaN,NaN,gig
5,gdp_data_quarterly,5,6.304001,9.383268e-06,NaN,NaN,gig
6,job_openings_data_monthly,5,6.280450,9.880915e-06,NaN,NaN,gig
7,job_openings_data_monthly,6,5.381146,1.819420e-05,NaN,NaN,gig
8,interest_rate_fedfunds_data_monthly,2,8.517013,2.176978e-04,NaN,NaN,gig
9,interest_rate_fedfunds_data_monthly,1,12.322673,4.710521e-04,NaN,NaN,gig


In [8]:

# Quarterly
try:
    for i, col in enumerate(gig_q_df.columns):
        if i == 0 and 'gig_q_gr_df' in gbl_lst:
            del gig_q_gr_df
        if col != 'target' and col not in columns_to_drop:
            gr = grangercausalitytests(gig_q_df[[col, 'target']], maxlag=1, verbose=False)
            gbl_lst = list(globals().keys())
            if 'gig_q_gr_df' not in gbl_lst:
                for k, v in gr.items():
                    if k == 1:
                        gig_q_gr_df = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                    else:
                        df2 = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                        gig_q_gr_df = pd.concat([gig_q_gr_df, df2], ignore_index=True)
            else:
                for k, v in gr.items():
                    df2 = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                    gig_q_gr_df = pd.concat([gig_q_gr_df, df2], ignore_index=True)
    gig_q_gr_df = gig_q_gr_df[gig_q_gr_df['p-value'] < quarterly_alpha].reset_index(drop=True)
    gig_q_gr_df = gig_q_gr_df.sort_values(by=['p-value', 'F-statistic'], ascending=[True, False]).reset_index(drop=True)

    iter_gig_q_gr_df = gig_q_gr_df[['column','lag']]
    for i, (col, lag) in enumerate(iter_gig_q_gr_df.itertuples(index=False, name=None)):
        gbl_lst = list(globals().keys())
        gr = grangercausalitytests(gig_q_df[['target', col]], maxlag=1, verbose=False)
        if i == 0:
            if 'gig_q_gr_df_rev' in gbl_lst:
                del gig_q_gr_df_rev
            gig_q_gr_df_rev = pd.DataFrame({'column':col,'lag':lag, 'F-statistic':gr[lag][0]['ssr_ftest'][0], 'p-value':gr[lag][0]['ssr_ftest'][1]}, index=[0])
        else:
            df2 = pd.DataFrame({'column':col,'lag':lag, 'F-statistic':gr[lag][0]['ssr_ftest'][0], 'p-value':gr[lag][0]['ssr_ftest'][1]}, index=[0])
            gig_q_gr_df_rev = pd.concat([gig_q_gr_df_rev, df2], ignore_index=True)
    gig_q_gr_df_rev = gig_q_gr_df_rev[gig_q_gr_df_rev['p-value'] < quarterly_alpha].reset_index(drop=True)
    gig_q_gr_df_rev = gig_q_gr_df_rev.sort_values(by=['p-value', 'F-statistic'], ascending=[True, False]).reset_index(drop=True)

    gig_q_gr_df_fin = gig_q_gr_df.merge(gig_q_gr_df_rev, on=['column', 'lag'], how='left', suffixes=('_forward', '_reverse')).reset_index(drop=True)
    gig_q_gr_df_fin = gig_q_gr_df_fin[gig_q_gr_df_fin['p-value_reverse'].isna()].reset_index(drop=True)
    gig_q_gr_df_fin['type'] = 'gig'

    for i, col in enumerate(sp500_q_df.columns):
        if i == 0 and 'sp500_q_gr_df' in gbl_lst:
            del sp500_q_gr_df
        if col != 'target' and col not in columns_to_drop:
            gr = grangercausalitytests(sp500_q_df[[col, 'target']], maxlag=1, verbose=False)
            gbl_lst = list(globals().keys())
            if 'sp500_q_gr_df' not in gbl_lst:
                for k, v in gr.items():
                    if k == 1:
                        sp500_q_gr_df = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                    else:
                        df2 = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                        sp500_q_gr_df = pd.concat([sp500_q_gr_df, df2], ignore_index=True)
            else:
                for k, v in gr.items():
                    df2 = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                    sp500_q_gr_df = pd.concat([sp500_q_gr_df, df2], ignore_index=True)
    sp500_q_gr_df = sp500_q_gr_df[sp500_q_gr_df['p-value'] < quarterly_alpha].reset_index(drop=True)
    sp500_q_gr_df = sp500_q_gr_df.sort_values(by=['p-value', 'F-statistic'], ascending=[True, False]).reset_index(drop=True)

    iter_sp500_q_gr_df = sp500_q_gr_df[['column','lag']]
    for i, (col, lag) in enumerate(iter_sp500_q_gr_df.itertuples(index=False, name=None)):
        gbl_lst = list(globals().keys())
        gr = grangercausalitytests(sp500_q_df[['target', col]], maxlag=1, verbose=False)
        if i == 0:
            if 'sp500_q_gr_df_rev' in gbl_lst:
                del sp500_q_gr_df_rev
            sp500_q_gr_df_rev = pd.DataFrame({'column':col,'lag':lag, 'F-statistic':gr[lag][0]['ssr_ftest'][0], 'p-value':gr[lag][0]['ssr_ftest'][1]}, index=[0])
        else:
            df2 = pd.DataFrame({'column':col,'lag':lag, 'F-statistic':gr[lag][0]['ssr_ftest'][0], 'p-value':gr[lag][0]['ssr_ftest'][1]}, index=[0])
            sp500_q_gr_df_rev = pd.concat([sp500_q_gr_df_rev, df2], ignore_index=True)
    sp500_q_gr_df_rev = sp500_q_gr_df_rev[sp500_q_gr_df_rev['p-value'] < quarterly_alpha].reset_index(drop=True)
    sp500_q_gr_df_rev = sp500_q_gr_df_rev.sort_values(by=['p-value', 'F-statistic'], ascending=[True, False]).reset_index(drop=True)

    sp500_q_gr_df_fin = sp500_q_gr_df.merge(sp500_q_gr_df_rev, on=['column', 'lag'], how='left', suffixes=('_forward', '_reverse')).reset_index(drop=True)
    sp500_q_gr_df_fin = sp500_q_gr_df_fin[sp500_q_gr_df_fin['p-value_reverse'].isna()].reset_index(drop=True)
    sp500_q_gr_df_fin['type'] = 'sp500'

    quarterly_gr_df_fin = pd.concat([gig_q_gr_df_fin, sp500_q_gr_df_fin], ignore_index=True)

except NameError:
    quarterly_gr_df_fin = None

quarterly_gr_df_fin


In [9]:
# Annual
try:
    for i, col in enumerate(gig_a_df.columns):
        if i == 0 and 'gig_a_gr_df' in gbl_lst:
            del gig_a_gr_df
        if col != 'target' and col not in columns_to_drop:
            gr = grangercausalitytests(gig_a_df[[col, 'target']], maxlag=1, verbose=False)
            gbl_lst = list(globals().keys())
            if 'gig_a_gr_df' not in gbl_lst:
                for k, v in gr.items():
                    if k == 1:
                        gig_a_gr_df = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                    else:
                        df2 = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                        gig_a_gr_df = pd.concat([gig_a_gr_df, df2], ignore_index=True)
            else:
                for k, v in gr.items():
                    df2 = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                    gig_a_gr_df = pd.concat([gig_a_gr_df, df2], ignore_index=True)
    gig_a_gr_df = gig_a_gr_df[gig_a_gr_df['p-value'] < annual_alpha].reset_index(drop=True)
    gig_a_gr_df = gig_a_gr_df.sort_values(by=['p-value', 'F-statistic'], ascending=[True, False]).reset_index(drop=True)

    iter_gig_a_gr_df = gig_a_gr_df[['column','lag']]
    for i, (col, lag) in enumerate(iter_gig_a_gr_df.itertuples(index=False, name=None)):
        gbl_lst = list(globals().keys())
        gr = grangercausalitytests(gig_a_df[['target', col]], maxlag=1, verbose=False)
        if i == 0:
            if 'gig_a_gr_df_rev' in gbl_lst:
                del gig_a_gr_df_rev
            gig_a_gr_df_rev = pd.DataFrame({'column':col,'lag':lag, 'F-statistic':gr[lag][0]['ssr_ftest'][0], 'p-value':gr[lag][0]['ssr_ftest'][1]}, index=[0])
        else:
            df2 = pd.DataFrame({'column':col,'lag':lag, 'F-statistic':gr[lag][0]['ssr_ftest'][0], 'p-value':gr[lag][0]['ssr_ftest'][1]}, index=[0])
            gig_a_gr_df_rev = pd.concat([gig_a_gr_df_rev, df2], ignore_index=True)
    gig_a_gr_df_rev = gig_a_gr_df_rev[gig_a_gr_df_rev['p-value'] < annual_alpha].reset_index(drop=True)
    gig_a_gr_df_rev = gig_a_gr_df_rev.sort_values(by=['p-value', 'F-statistic'], ascending=[True, False]).reset_index(drop=True)

    gig_a_gr_df_fin = gig_a_gr_df.merge(gig_a_gr_df_rev, on=['column', 'lag'], how='left', suffixes=('_forward', '_reverse')).reset_index(drop=True)
    gig_a_gr_df_fin = gig_a_gr_df_fin[gig_a_gr_df_fin['p-value_reverse'].isna()].reset_index(drop=True)
    gig_a_gr_df_fin['type'] = 'gig'

    for i, col in enumerate(sp500_a_df.columns):
        if i == 0 and 'sp500_a_gr_df' in gbl_lst:
            del sp500_a_gr_df
        if col != 'target' and col not in columns_to_drop:
            gr = grangercausalitytests(sp500_a_df[[col, 'target']], maxlag=1, verbose=False)
            gbl_lst = list(globals().keys())
            if 'sp500_a_gr_df' not in gbl_lst:
                for k, v in gr.items():
                    if k == 1:
                        sp500_a_gr_df = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                    else:
                        df2 = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                        sp500_a_gr_df = pd.concat([sp500_a_gr_df, df2], ignore_index=True)
            else:
                for k, v in gr.items():
                    df2 = pd.DataFrame({'column':col,'lag':k, 'F-statistic':v[0]['ssr_ftest'][0], 'p-value':v[0]['ssr_ftest'][1]}, index=[0])
                    sp500_a_gr_df = pd.concat([sp500_a_gr_df, df2], ignore_index=True)
    sp500_a_gr_df = sp500_a_gr_df[sp500_a_gr_df['p-value'] < annual_alpha].reset_index(drop=True)
    sp500_a_gr_df = sp500_a_gr_df.sort_values(by=['p-value', 'F-statistic'], ascending=[True, False]).reset_index(drop=True)

    iter_sp500_a_gr_df = sp500_a_gr_df[['column','lag']]

    for i, (col, lag) in enumerate(iter_sp500_a_gr_df.itertuples(index=False, name=None)):
        gbl_lst = list(globals().keys())
        gr = grangercausalitytests(sp500_a_df[['target', col]], maxlag=1, verbose=False)
        if i == 0:
            if 'sp500_a_gr_df_rev' in gbl_lst:
                del sp500_a_gr_df_rev
            sp500_a_gr_df_rev = pd.DataFrame({'column':col,'lag':lag, 'F-statistic':gr[lag][0]['ssr_ftest'][0], 'p-value':gr[lag][0]['ssr_ftest'][1]}, index=[0])
        else:
            df2 = pd.DataFrame({'column':col,'lag':lag, 'F-statistic':gr[lag][0]['ssr_ftest'][0], 'p-value':gr[lag][0]['ssr_ftest'][1]}, index=[0])
            sp500_a_gr_df_rev = pd.concat([sp500_a_gr_df_rev, df2], ignore_index=True)
    sp500_a_gr_df_rev = sp500_a_gr_df_rev[sp500_a_gr_df_rev['p-value'] < annual_alpha].reset_index(drop=True)
    sp500_a_gr_df_rev = sp500_a_gr_df_rev.sort_values(by=['p-value', 'F-statistic'], ascending=[True, False]).reset_index(drop=True)

    sp500_a_gr_df_fin = sp500_a_gr_df.merge(sp500_a_gr_df_rev, on=['column', 'lag'], how='left', suffixes=('_forward', '_reverse')).reset_index(drop=True)
    sp500_a_gr_df_fin = sp500_a_gr_df_fin[sp500_a_gr_df_fin['p-value_reverse'].isna()].reset_index(drop=True)
    sp500_a_gr_df_fin['type'] = 'sp500'

    annual_gr_df_fin = pd.concat([gig_a_gr_df_fin, sp500_a_gr_df_fin], ignore_index=True)

except NameError:
    annual_gr_df_fin = None

annual_gr_df_fin
